---
## 📦 Cell 1 — Check & Install Dependencies

In [ ]:
import subprocess, sys

print('=' * 60)
print('📦 DEPENDENCY CHECK')
print('=' * 60)

required = [
    ('tensorflow', 'tensorflow'),
    ('cv2',        'opencv-python'),
    ('numpy',      'numpy'),
    ('matplotlib', 'matplotlib'),
]

missing = []
for module, package in required:
    try:
        __import__(module)
        print(f'   ✅ {package} — already installed')
    except ImportError:
        print(f'   ⚠️  {package} — NOT found, will install...')
        missing.append(package)

if missing:
    print(f'\n⬇️  Installing: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
    print('   ✅ All missing packages installed!')
else:
    print('\n✅ All dependencies are already satisfied!')

print('\nPython version:', sys.version)

---
## 🔧 Cell 2 — Imports & Configuration

In [ ]:
print('Importing standard libraries...')
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
print('   ✅ os, sys, numpy, matplotlib, opencv')

print('Importing TensorFlow...')
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from tensorflow.keras.applications import MobileNetV2 # type: ignore
from tensorflow.keras import layers, models # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from tensorflow.keras.models import load_model # type: ignore
print(f'   ✅ TensorFlow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'   • GPU available: {"Yes — " + str(gpus) if gpus else "No (running on CPU)"}')

# ── Paths (relative to notebooks/ directory) ──────────────────
DATA_PATH  = '../data/train'
MODEL_PATH = '../models/fruit_model.h5'

# ── Settings ──────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# ── Class names (must match folder names EXACTLY) ─────────────
CLASS_NAMES = [
    'Apple',
    'Banana',
    'Blackberry',
    'Cherry',
    'Mango',
    'Raspberry',
    'Strawberry'
]

print(f'\n✅ All imports successful — ready to go!')
print(f'   Data path  : {os.path.abspath(DATA_PATH)}')
print(f'   Model path : {os.path.abspath(MODEL_PATH)}')

---
## 🏋️ Cell 3 — Train the Model

Trains MobileNetV2 on your dataset. Training will automatically stop early if validation loss stops improving (patience = 3 epochs).

> ⏭️ **Auto-skipped** if `fruit_model.h5` already exists.  
> ⏳ **Expected time**: ~2–10 minutes depending on your hardware.

In [ ]:
history = None   # will hold training history (None if we skip training)

# ── Check if model already exists ─────────────────────────────
if os.path.exists(MODEL_PATH):
    print('⏭️  Model already exists — skipping training.')
    print(f'   Found at: {os.path.abspath(MODEL_PATH)}')
else:
    # ── Verify dataset ────────────────────────────────────────
    print('🗂️  Verifying dataset structure...')
    if not os.path.isdir(DATA_PATH):
        raise FileNotFoundError(
            f'❌ Training data folder not found: {DATA_PATH}\n'
            'Please ensure the /data/train/ directory exists with class sub-folders.'
        )

    class_folders = sorted(os.listdir(DATA_PATH))
    print(f'   Found {len(class_folders)} class folder(s): {class_folders}')
    for folder in class_folders:
        folder_path = os.path.join(DATA_PATH, folder)
        count = len(os.listdir(folder_path))
        print(f'   • {folder:<20} — {count} images')

    print('\n' + '=' * 60)
    print('🚀 FRUIT CLASSIFICATION MODEL - TRAINING STARTED')
    print('=' * 60)

    print(f'\n📋 Training Configuration:')
    print(f'   • Image Size  : {IMG_SIZE[0]}x{IMG_SIZE[1]} pixels')
    print(f'   • Batch Size  : {BATCH_SIZE}')
    print(f'   • Max Epochs  : 10')
    print(f'   • Data Path   : {DATA_PATH}')
    print(f'   • Model Output: {MODEL_PATH}')

    # ── Data generators ───────────────────────────────────────
    print('\n📂 Step 1 — Setting up Image Data Generators...')
    print('   Applying augmentations: rescale, rotation(20°), zoom(20%), horizontal flip')
    print('   80% data → Training  |  20% data → Validation')

    datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=0.2
    )

    print('\n   Loading TRAINING data from disk...')
    train_data = datagen.flow_from_directory(
        DATA_PATH,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    print('\n   Loading VALIDATION data from disk...')
    val_data = datagen.flow_from_directory(
        DATA_PATH,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )

    num_classes = train_data.num_classes
    detected_classes = list(train_data.class_indices.keys())

    print(f'\n✅ Data loaded successfully!')
    print(f'   • Classes found ({num_classes}): {detected_classes}')
    print(f'   • Training samples   : {train_data.samples}')
    print(f'   • Validation samples : {val_data.samples}')
    print(f'   • Training batches   : {len(train_data)}')
    print(f'   • Validation batches : {len(val_data)}')

    # ── Build model ───────────────────────────────────────────
    print('\n🧠 Step 2 — Loading MobileNetV2 Base Model (pretrained on ImageNet)...')
    print('   Downloading weights if not cached (this may take a moment)...')

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    print(f'   ✅ MobileNetV2 loaded — {len(base_model.layers)} layers (all frozen)')
    print('   💡 We only train the new classification head on top.')

    print('\n🔧 Step 3 — Building Custom Classification Head...')
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    print(f'   Architecture:')
    print(f'     MobileNetV2 (frozen) → GlobalAveragePooling2D → Dense(128, ReLU) → Dense({num_classes}, Softmax)')

    # ── Compile ───────────────────────────────────────────────
    print('\n⚙️  Step 4 — Compiling Model...')
    print('   Optimizer: Adam  |  Loss: Categorical Crossentropy  |  Metric: Accuracy')
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    total_params     = model.count_params()
    trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
    print(f'   Total parameters     : {total_params:,}')
    print(f'   Trainable parameters : {trainable_params:,}')

    # ── Train ─────────────────────────────────────────────────
    print('\n🏋️  Step 5 — Training the Model...')
    print('   Early stopping enabled: patience=3 (monitors val_loss)')
    print('   Training will stop early if validation loss stops improving.\n')
    print('-' * 60)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=10,
        callbacks=[early_stop],
        verbose=1
    )

    epochs_trained  = len(history.history['accuracy'])
    final_train_acc = history.history['accuracy'][-1] * 100
    final_val_acc   = history.history['val_accuracy'][-1] * 100
    best_val_loss   = min(history.history['val_loss'])

    print('-' * 60)
    print(f'\n🎉 Training Complete!')
    print(f'   • Epochs trained      : {epochs_trained}/10')
    print(f'   • Final Train Acc     : {final_train_acc:.2f}%')
    print(f'   • Final Val Acc       : {final_val_acc:.2f}%')
    print(f'   • Best Val Loss       : {best_val_loss:.4f}')

    # ── Save model ────────────────────────────────────────────
    print(f'\n💾 Step 6 — Saving Model to \'{MODEL_PATH}\'...')
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    model.save(MODEL_PATH)
    size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f'   ✅ Model saved! File size: {size_mb:.1f} MB')

    print('\n' + '=' * 60)
    print('✅ TRAINING PIPELINE COMPLETE')
    print('=' * 60)
    print('\n🎉 Model is trained and saved. Proceed to Cell 4 to view training curves.')

---
## 📊 Cell 4 — Plot Training History

Visualise how accuracy and loss evolved over each epoch for both training and validation sets.

> ℹ️ Skipped automatically if you loaded a pre-trained model.

In [ ]:
if history is None:
    print('ℹ️  No training history to plot (model was pre-loaded or training was skipped).')
else:
    print('Plotting training history...')

    epochs = range(1, len(history.history['accuracy']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Model Training History', fontsize=16, fontweight='bold')

    # Accuracy
    axes[0].plot(epochs, history.history['accuracy'],     'b-o', label='Train Accuracy',      linewidth=2, markersize=5)
    axes[0].plot(epochs, history.history['val_accuracy'], 'r-o', label='Validation Accuracy', linewidth=2, markersize=5)
    axes[0].set_title('Accuracy over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)
    axes[0].set_ylim([0, 1.05])

    # Loss
    axes[1].plot(epochs, history.history['loss'],     'b-o', label='Train Loss',      linewidth=2, markersize=5)
    axes[1].plot(epochs, history.history['val_loss'], 'r-o', label='Validation Loss', linewidth=2, markersize=5)
    axes[1].set_title('Loss over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    # Summary stats
    best_epoch    = np.argmax(history.history['val_accuracy']) + 1
    best_val_acc  = max(history.history['val_accuracy']) * 100
    best_val_loss = min(history.history['val_loss'])

    print(f'\n📈 Training Summary:')
    print(f'   • Best Epoch          : {best_epoch}')
    print(f'   • Best Val Accuracy   : {best_val_acc:.2f}%')
    print(f'   • Best Val Loss       : {best_val_loss:.4f}')
    print(f'   • Total Epochs Ran    : {len(epochs)}')

    print(f'\n📈 Interpretation tip:')
    print('   • If train accuracy >> val accuracy  → model may be overfitting')
    print('   • If both curves are close           → good generalisation')
    print('   • If val accuracy is still rising    → could train for more epochs')

---
## 🔍 Cell 5 — Predict on a New Image

A **file-chooser window** will open automatically. Select any `.jpg`, `.png`, or `.jpeg` fruit image from your computer.  
The model will predict the fruit type and display a confidence bar chart.

> 🔁 You can re-run this cell as many times as you like to predict different images.

In [ ]:
import tkinter as tk
from tkinter import filedialog

# ──────────────────────────────────────────────────────────────
# Step 1 — Load the trained model
# ──────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('🔍 FRUIT CLASSIFICATION — PREDICTION MODULE')
print('=' * 60)

print(f"\n📦 Step 1 — Loading trained model from '{MODEL_PATH}'...")

if not os.path.exists(MODEL_PATH):
    print(f'   ❌ ERROR: Model file not found at \'{MODEL_PATH}\'')
    print('   💡 Please run Cell 3 first to train and save the model.')
else:
    loaded_model = load_model(MODEL_PATH)
    size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f'   ✅ Model loaded successfully! ({size_mb:.1f} MB)')
    print(f'   • Input shape  : {loaded_model.input_shape}')
    print(f'   • Output shape : {loaded_model.output_shape}  ({len(CLASS_NAMES)} classes)')

    # ──────────────────────────────────────────────────────────
    # Step 2 — Open file-chooser GUI window
    # ──────────────────────────────────────────────────────────
    print('\n🖼️  Step 2 — Selecting Image...')
    print('   Opening file chooser window — please select a fruit image.')

    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        img_path = filedialog.askopenfilename(
            title='Select a Fruit Image',
            filetypes=[
                ('Image Files', '*.jpg *.jpeg *.png *.bmp *.webp'),
                ('All Files', '*.*')
            ]
        )
        root.destroy()
    except Exception as e:
        print(f'   ⚠️  GUI dialog unavailable ({e}). Falling back to text input.')
        img_path = input('   👉 Enter full image path manually: ').strip()

    if not img_path:
        print('   ⚠️  No file selected. Re-run this cell to try again.')
    elif not os.path.exists(img_path):
        print(f'   ❌ ERROR: Image file not found at \'{img_path}\'')
    else:
        print(f'   ✅ File selected: {img_path}')

        # ──────────────────────────────────────────────────────
        # Step 3 — Preprocess the image
        # ──────────────────────────────────────────────────────
        print(f'\n⚙️  Step 3 — Preprocessing Image...')
        print(f'   Reading: {img_path}')

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print('   ❌ ERROR: Could not read image. File may be corrupt or unsupported.')
        else:
            h, w = img_bgr.shape[:2]
            print(f'   • Original size : {w}x{h} pixels')

            img_rgb        = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_resized    = cv2.resize(img_rgb, IMG_SIZE)
            img_normalized = img_resized / 255.0
            img_batch      = np.expand_dims(img_normalized, axis=0)

            print('   • Color space   : BGR → RGB ✅')
            print(f'   • Resized to    : {IMG_SIZE[0]}x{IMG_SIZE[1]} pixels ✅')
            print('   • Normalized    : pixel values scaled to [0.0, 1.0] ✅')
            print(f'   • Batch shape   : {img_batch.shape}  (1 image, {IMG_SIZE[0]}x{IMG_SIZE[1]}, 3 channels) ✅')

            # ──────────────────────────────────────────────────
            # Step 4 — Run inference
            # ──────────────────────────────────────────────────
            print(f'\n🤖 Step 4 — Running Model Inference...')
            print('   Passing preprocessed image through MobileNetV2 network...')

            predictions     = loaded_model.predict(img_batch, verbose=0)
            all_probs       = predictions[0]
            predicted_idx   = np.argmax(all_probs)
            predicted_class = CLASS_NAMES[predicted_idx]
            confidence      = all_probs[predicted_idx] * 100

            print(f'\n   📊 Raw Probabilities:')
            for i, (cls, prob) in enumerate(zip(CLASS_NAMES, all_probs)):
                marker = ' ◀ PREDICTED' if i == predicted_idx else ''
                bar    = '█' * int(prob * 30)
                print(f'      {cls:<15} {bar:<32} {prob * 100:5.1f}%{marker}')

            print(f'\n{"=" * 60}')
            print(f'  🍎 Predicted Fruit : {predicted_class}')
            print(f'  📊 Confidence      : {confidence:.2f}%')
            if confidence >= 80:
                print(f'  🟢 Confidence Level: HIGH — very likely correct')
            elif confidence >= 50:
                print(f'  🟡 Confidence Level: MODERATE — probably correct')
            else:
                print(f'  🔴 Confidence Level: LOW — model is uncertain')
            print(f'{"=" * 60}')

            # ──────────────────────────────────────────────────
            # Step 5 — Visualise the result
            # ──────────────────────────────────────────────────
            print('\n🖼️  Generating Prediction Visualization...')

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle('Fruit Classification Result', fontsize=16, fontweight='bold')

            # Image panel
            axes[0].imshow(img_rgb)
            axes[0].set_title(
                f'Predicted: {predicted_class}\nConfidence: {confidence:.2f}%',
                fontsize=13, color='green', fontweight='bold'
            )
            axes[0].axis('off')

            # Probability bar chart
            colors = ['#2ecc71' if c == predicted_class else '#3498db' for c in CLASS_NAMES]
            bars   = axes[1].barh(CLASS_NAMES, all_probs * 100, color=colors)
            axes[1].set_xlabel('Confidence (%)')
            axes[1].set_title('Class Probabilities')
            axes[1].set_xlim([0, 110])
            axes[1].grid(True, axis='x', linestyle='--', alpha=0.5)

            for bar, prob in zip(bars, all_probs * 100):
                axes[1].text(
                    bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f'{prob:.1f}%', va='center', fontsize=9
                )

            plt.tight_layout()
            plt.show()

            print('   ✅ Visualization displayed.')
            print(f'\n🏆 Final Answer: The fruit is a "{predicted_class}" with {confidence:.2f}% confidence.')